# 新版 LangChain：记录最近 k 条（窗口）Memory

这一节对应老版本的 `ConversationBufferWindowMemory`。
在新版 LCEL / Runnable 体系里，推荐两种做法：

1. **窗口注入（推荐）**：用 `MessagesPlaceholder(n_messages=...)` 控制每次送给模型的历史条数。
2. **严格裁剪（可选）**：在 history 存储层主动截断，只保留最近 k 轮。

## 新旧版本对应关系

- 旧版：`ConversationBufferWindowMemory(k=K)`
- 新版：`RunnableWithMessageHistory` + `MessagesPlaceholder(variable_name="history", n_messages=K*2)`

说明：旧版 `k` 通常按“最近 k 轮对话（人类+AI）”理解，因此常对应 `2*k` 条消息。

In [1]:
# 1) 环境准备
import os
from typing import Dict

from dotenv import load_dotenv
from pydantic import SecretStr
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

load_dotenv(override=True)

model_name = os.getenv("MODEL")
model_base_url = os.getenv("BASE_URL")
model_api_key = os.getenv("API_KEY")
api_key_value = SecretStr(model_api_key) if model_api_key else None

chat_llm = ChatOpenAI(
    model=model_name or "gpt-4o-mini",
    base_url=model_base_url or None,
    api_key=api_key_value,
    temperature=0,
 )

print("✅ 模型初始化完成")

✅ 模型初始化完成


In [2]:
# 2) 方案A：窗口注入（推荐）
WINDOW_K = 1  # 最近 2 轮
WINDOW_MESSAGES = WINDOW_K * 2  # 约等于 4 条消息（人类+AI）

prompt_window = ChatPromptTemplate.from_messages([
    ("system", "你是一个简洁的中文助理。"),
    MessagesPlaceholder(variable_name="history", n_messages=WINDOW_MESSAGES),
    ("human", "{input}"),
])

base_chain_window = prompt_window | chat_llm

store_window: Dict[str, ChatMessageHistory] = {}

def get_window_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store_window:
        store_window[session_id] = ChatMessageHistory()
    return store_window[session_id]

chain_with_window_memory = RunnableWithMessageHistory(
    base_chain_window,
    get_window_history,
    input_messages_key="input",
    history_messages_key="history",
)

print("✅ 方案A链创建完成")

✅ 方案A链创建完成


In [3]:
# 3) 运行多轮并观察：存储是全量，但注入给模型的是最近窗口
session_id = "window_user_001"
questions = [
    "清朝持续了多久？",
    "1+1等于几？",
    "我今天想学langchain。",
    "明朝持续了多久？", 
    "我第一个问题是什么"
]

for i, q in enumerate(questions, start=1):
    ans = chain_with_window_memory.invoke(
        {"input": q},
        config={"configurable": {"session_id": session_id}},
    )
    print(f"第{i}轮回答: {ans.content}")

history_all = store_window[session_id].messages
print("\n总存储消息条数:", len(history_all))

preview_messages = prompt_window.format_messages(
    input="测试窗口注入",
    history=history_all,
 )
injected_history_count = len(preview_messages) - 2  # system + 当前human
print("本次注入给模型的历史条数:", injected_history_count)
print("预期窗口条数:", WINDOW_MESSAGES)

第1轮回答: 清朝通常算作 **1636 年建立，1912 年结束**，一共 **276 年**。

如果按它 **入关并开始统治全国的 1644 年** 算，到 1912 年则是 **268 年**。
第2轮回答: 1 + 1 = 2。
第3轮回答: 好呀，LangChain 适合用来快速把大模型接到你的应用里。

你可以这样学：

1. **先理解它是干什么的**
   - 把 LLM 接到业务里
   - 做提示词管理、工具调用、RAG、Agent、记忆等

2. **先学基础概念**
   - LLM / ChatModel
   - PromptTemplate
   - Chain
   - Retriever / VectorStore
   - Tool / Agent
   - Memory

3. **动手顺序建议**
   - Hello World：调用一个模型
   - Prompt 模板：输入变量 + 输出格式
   - 链式调用：把多个步骤串起来
   - RAG：加载文档 → 向量检索 → 生成答案
   - 工具调用：让模型查天气、查数据库
   - Agent：让模型自己决定用什么工具

4. **建议先掌握的技术栈**
   - Python
   - 基本 API 调用
   - 向量数据库基础
   - Embedding 基础
   - JSON / 正则 / 文本处理

如果你愿意，我可以直接带你：
- **从 0 写一个 LangChain 入门示例**
- 或者给你一份 **3 天学习路线**
- 也可以按你的语言环境讲：**Python / JavaScript**  

你现在更想学哪一种？
第4轮回答: 明朝持续了 **276年**。

- **起始**：1368年
- **结束**：1644年

如果你想，我也可以顺便给你列一下明朝的重要历史节点。
第5轮回答: 你第一个问题是：**“明朝持续了多久？”**

总存储消息条数: 10
本次注入给模型的历史条数: 2
预期窗口条数: 2


## 方案B：严格只保留最近 k 轮（可选）

如果你不仅想“注入窗口”，还想“存储也只留最近 k 轮”，可以在 history 层做裁剪。

In [4]:
# 4) 严格裁剪：存储层只保留最近 k 轮
MAX_MESSAGES = WINDOW_K * 2
store_strict: Dict[str, ChatMessageHistory] = {}

def _trim_messages(history: ChatMessageHistory, max_messages: int) -> None:
    if len(history.messages) > max_messages:
        history.messages = history.messages[-max_messages:]

def get_strict_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store_strict:
        store_strict[session_id] = ChatMessageHistory()
    history = store_strict[session_id]
    _trim_messages(history, MAX_MESSAGES)  # 读之前先裁剪
    return history

prompt_strict = ChatPromptTemplate.from_messages([
    ("system", "你是一个简洁的中文助理。"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

base_chain_strict = prompt_strict | chat_llm
chain_with_strict_memory = RunnableWithMessageHistory(
    base_chain_strict,
    get_strict_history,
    input_messages_key="input",
    history_messages_key="history",
)

def invoke_with_strict_trim(user_input: str, sid: str):
    result = chain_with_strict_memory.invoke(
        {"input": user_input},
        config={"configurable": {"session_id": sid}},
    )
    _trim_messages(store_strict[sid], MAX_MESSAGES)  # 写之后再裁剪
    return result

print("✅ 方案B链创建完成")

✅ 方案B链创建完成


In [5]:
# 5) 验证严格裁剪：消息条数不会超过 2*k
strict_session = "strict_user_001"
strict_questions = [
    "第一轮：我叫小严",
    "第二轮：我来自北京",
    "第三轮：我喜欢篮球",
    "第四轮：我在学LangChain",
    "第五轮：你还记得我叫什么吗？",
]

for i, q in enumerate(strict_questions, start=1):
    resp = invoke_with_strict_trim(q, strict_session)
    current_len = len(store_strict[strict_session].messages)
    print(f"第{i}轮 -> 当前存储条数: {current_len} | 回答: {resp.content}")

print("\n最终存储条数:", len(store_strict[strict_session].messages))
print("最大允许条数:", MAX_MESSAGES)

第1轮 -> 当前存储条数: 2 | 回答: 你好，小严，很高兴认识你。
第2轮 -> 当前存储条数: 2 | 回答: 你好，小严，来自北京，很高兴认识你。
第3轮 -> 当前存储条数: 2 | 回答: 太好了，篮球很有意思！你平时更喜欢打球，还是看比赛？
第4轮 -> 当前存储条数: 2 | 回答: 不错，LangChain 很适合做 LLM 应用开发。你现在是在学基础用法，还是在做具体项目？
第5轮 -> 当前存储条数: 2 | 回答: 我还不知道你的名字呢，你可以告诉我，我就记住这个对话里称呼你。

最终存储条数: 2
最大允许条数: 2


## 小结

- 想替代 `ConversationBufferWindowMemory`：优先用 **方案A（`n_messages`）**，最轻量。
- 想控制存储成本：用 **方案B（严格裁剪）**。
- 两种方案都可以和 `RunnableWithMessageHistory`、Redis 历史存储一起使用。